# Turkey Earthquake Analytics
## Earthquake data pipeline and exploration — portfolio case study

**Goal:** Turn a public earthquake catalogue into a clean, documented dataset that can be explored in SQL and Power BI.

The notebook demonstrates a reproducible ETL workflow using real USGS catalogue records. The query covers a rectangle around Turkey and can include neighboring countries and sea areas. Facility examples remain synthetic and optional. This project does not predict earthquakes or assess damage or safety.

**Skills demonstrated:** API ingestion, JSON normalization, data-quality rules, Turkish-text normalization, dimensional modeling, SQL analysis, geospatial calculations, audit manifests, and a Power BI handoff.


## 1. Setup
Requires Python 3.11+ with pandas and NumPy. From the project directory:
`python -m pip install -r requirements.txt`, then `jupyter lab`.

The transformation functions are embedded below so the notebook is self-contained.
`pipeline.py` contains the same implementation for unit tests and reuse.


In [ ]:
"""Earthquake ingestion and facility screening. No damage or prediction model."""
from datetime import datetime, timedelta, timezone
from pathlib import Path
import hashlib
import json
import logging
import math
import sqlite3
import time
import unicodedata
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import Request, urlopen

import numpy as np
import pandas as pd

LOG = logging.getLogger(__name__)
AFAD_URL = 'https://deprem.afad.gov.tr/apiv2/event/filter'
PROVINCE_URL = 'https://turkiyeapi.dev/api/v1/provinces'
USGS_URL = 'https://earthquake.usgs.gov/fdsnws/event/1/'


def usgs_records(payload):
    """Adapt GeoJSON; coordinates are longitude, latitude, depth in km."""
    if not isinstance(payload, dict) or payload.get('type') != 'FeatureCollection' or not isinstance(payload.get('features'), list):
        raise ValueError('Expected USGS GeoJSON FeatureCollection.')
    records = []
    for feature in payload['features']:
        p, geometry = feature.get('properties') or {}, feature.get('geometry') or {}
        coordinates = geometry.get('coordinates')
        if geometry.get('type') != 'Point' or not isinstance(coordinates, list) or len(coordinates) != 3:
            raise ValueError('USGS event has invalid Point geometry.')
        def epoch(value):
            return None if value is None else pd.to_datetime(value, unit='ms', utc=True).isoformat()
        records.append({'eventID': feature.get('id'), 'date': epoch(p.get('time')),
            'longitude': coordinates[0], 'latitude': coordinates[1], 'depth': coordinates[2],
            'magnitude': p.get('mag'), 'type': p.get('magType'), 'location': p.get('place'),
            'province': None, 'country': None, 'lastUpdateDate': epoch(p.get('updated')),
            'event_source': 'USGS', 'source_event_url': p.get('url')})
    return records


def fetch_usgs_events(start, end, raw_dir, min_magnitude=4.0, limit=1000, fetcher=None):
    """UTC query with count reconciliation, subdivision and Turkey-region bbox.

    Bounding box includes neighboring countries and sea areas. Counts establish
    retrieval coverage for the query, not completeness of the seismic catalogue.
    """
    fetcher = fetcher or fetch_json
    if start.tzinfo or end.tzinfo or start >= end or not 2 <= limit <= 20000:
        raise ValueError('Use naive UTC start < end and limit 2..20000.')
    if start.microsecond or end.microsecond:
        raise ValueError('Use whole-second boundaries.')
    def window(lo, hi):
        params = {'format': 'geojson', 'starttime': lo.isoformat(), 'endtime': hi.isoformat(),
            'minlatitude': 35, 'maxlatitude': 43, 'minlongitude': 25, 'maxlongitude': 45,
            'minmagnitude': min_magnitude, 'eventtype': 'earthquake'}
        count_payload = fetcher(USGS_URL + 'count', params, raw_dir)
        if not isinstance(count_payload, dict) or not isinstance(count_payload.get('count'), int) or count_payload['count'] < 0:
            raise ValueError('Invalid USGS count response.')
        count = count_payload['count']
        if count > limit:
            seconds = int((hi-lo).total_seconds())
            if seconds <= 1:
                raise RuntimeError('USGS window exceeds cap; cannot establish coverage.')
            mid = lo + timedelta(seconds=seconds // 2)
            return window(lo, mid) + window(mid, hi)
        if count == 0:
            return []
        payload = fetcher(USGS_URL + 'query', {**params, 'limit': limit, 'orderby': 'time-asc'}, raw_dir)
        rows = usgs_records(payload)
        if len(rows) != count or len({r['eventID'] for r in rows}) != count:
            raise RuntimeError('USGS count/query mismatch or duplicate IDs; rerun to reconcile.')
        return rows
    rows = []
    cursor = start
    while cursor < end:
        stop = min(cursor+timedelta(days=1), end)
        rows.extend(window(cursor, stop))
        cursor = stop
    return rows


def fetch_json(url, params, raw_dir, attempts=3):
    """Timeout, bounded retries, TLS verification, and immutable raw snapshots."""
    request_url = url + '?' + urlencode(params)
    for attempt in range(attempts):
        try:
            request = Request(request_url, headers={'User-Agent': 'EarthquakeAnalyticsPortfolio/1.0'})
            with urlopen(request, timeout=30) as response:
                payload = json.load(response)
            break
        except HTTPError as exc:
            if exc.code not in (429, 500, 502, 503, 504) or attempt == attempts - 1:
                raise
            time.sleep(min(2 ** attempt, 8))
        except URLError:
            if attempt == attempts - 1:
                raise
            time.sleep(min(2 ** attempt, 8))
    envelope = {'url': request_url, 'retrieved_at_utc': datetime.now(timezone.utc).isoformat(),
                'payload': payload}
    raw_dir = Path(raw_dir)
    raw_dir.mkdir(parents=True, exist_ok=True)
    key = hashlib.sha256(request_url.encode()).hexdigest()[:16]
    stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%f')
    (raw_dir / f'{stamp}_{key}.json').write_text(json.dumps(envelope, ensure_ascii=False), encoding='utf-8')
    return payload


def fetch_events(start, end, raw_dir, source_timezone, fetcher=fetch_json, limit=1000):
    """Fetch daily windows; split saturated windows instead of silently truncating.

    start/end are naive source-clock datetimes; end is exclusive. API time-zone
    semantics must be verified by the caller before running live. Adjacent windows
    may overlap at endpoints; normalization resolves repeated event IDs.
    """
    if not source_timezone or start.tzinfo or end.tzinfo or start >= end or limit < 2:
        raise ValueError('Provide a verified source timezone, naive start < end, and limit >= 2.')
    if start.microsecond or end.microsecond:
        raise ValueError('Query boundaries must be whole seconds.')
    def window(lo, hi):
        params = {'start': lo.isoformat(sep=' '), 'end': hi.isoformat(sep=' '), 'limit': limit}
        rows = fetcher(AFAD_URL, params, raw_dir)
        if not isinstance(rows, list) or any(not isinstance(x, dict) for x in rows):
            raise ValueError('AFAD schema changed: expected a list of event objects.')
        if len(rows) >= limit:
            seconds = int((hi - lo).total_seconds())
            if seconds <= 1:
                raise RuntimeError('Saturated one-second window: completeness cannot be established.')
            mid = lo + timedelta(seconds=seconds // 2)
            return window(lo, mid) + window(mid, hi)
        LOG.info('Fetched %s records for %s to %s', len(rows), lo, hi)
        return rows
    rows = []
    cursor = start
    while cursor < end:
        stop = min(cursor + timedelta(days=1), end)
        rows.extend(window(cursor, stop))
        cursor = stop
    return rows


def name_key(value):
    if value is None or pd.isna(value):
        return ''
    value = str(value).strip().replace('İ', 'I').replace('ı', 'i').casefold()
    return ''.join(c for c in unicodedata.normalize('NFKD', value) if not unicodedata.combining(c))


def province_tables(payload):
    """Separate grains; district IDs and names never overwrite province columns."""
    if not isinstance(payload, dict) or not isinstance(payload.get('data'), list):
        raise ValueError('Expected TurkiyeAPI object with a data list.')
    provinces, districts = [], []
    for p in payload['data']:
        provinces.append({'province_id': int(p['id']), 'province_name': p['name'],
                          'province_key': name_key(p['name']), 'region': (p.get('region') or {}).get('en')})
        for d in p.get('districts') or []:
            districts.append({'district_id': int(d['id']), 'district_name': d['name'],
                              'province_id': int(p['id'])})
    provinces = pd.DataFrame(provinces, columns=['province_id', 'province_name', 'province_key', 'region'])
    districts = pd.DataFrame(districts, columns=['district_id', 'district_name', 'province_id'])
    if provinces.empty or (provinces.province_id <= 0).any() or (provinces.province_key == '').any():
        raise ValueError('Invalid/empty province reference data.')
    if provinces.province_id.duplicated().any() or provinces.province_key.duplicated().any():
        raise ValueError('Ambiguous province IDs or normalized names.')
    if districts.district_id.duplicated().any():
        raise ValueError('Duplicate district IDs.')
    unknown = pd.DataFrame([{'province_id': 0, 'province_name': 'Unmatched / outside reference',
                             'province_key': '', 'region': None}])
    return pd.concat([unknown, provinces], ignore_index=True), districts


def parse_time(value, source_timezone):
    try:
        ts = pd.Timestamp(value)
        if pd.isna(ts):
            return pd.NaT
        if ts.tzinfo is None:
            ts = ts.tz_localize(source_timezone, ambiguous='raise', nonexistent='raise')
        return ts.tz_convert('UTC')
    except (ValueError, TypeError):
        return pd.NaT


def normalize_events(records, provinces, source_timezone, start=None, end=None, allow_negative_depth=False):
    """Quarantine invalid rows; use latest source update for duplicate IDs.

    Negative magnitude is valid. Missing magnitude/depth is retained and counted,
    never replaced with zero. Unknown province uses explicit key 0, not a guessed
    nearest province. A source province label is not a polygon spatial assignment.
    """
    columns = ['eventID', 'date', 'latitude', 'longitude', 'magnitude', 'depth',
               'type', 'province', 'country', 'location', 'lastUpdateDate', 'event_source', 'source_event_url']
    frame = pd.DataFrame(records)
    required = {'eventID', 'date', 'latitude', 'longitude'}
    if len(frame) and not required.issubset(frame.columns):
        raise ValueError(f'Missing event fields: {required - set(frame.columns)}')
    frame = frame.reindex(columns=columns).copy()
    frame['event_id'] = frame.eventID.astype('string').str.strip()
    frame['event_time_utc'] = pd.to_datetime(frame.date.map(lambda v: parse_time(v, source_timezone)), utc=True)
    frame['updated_at'] = pd.to_datetime(frame.lastUpdateDate.map(lambda v: parse_time(v, source_timezone)), utc=True)
    reasons = pd.Series('', index=frame.index)
    def flag(mask, reason):
        reasons.loc[mask.fillna(True)] += reason + ';'
    flag(frame.event_id.isna() | frame.event_id.eq(''), 'missing_event_id')
    flag(frame.event_time_utc.isna(), 'invalid_timestamp')
    for column in ['latitude', 'longitude', 'magnitude', 'depth']:
        source = frame[column]
        converted = pd.to_numeric(source, errors='coerce')
        missing = source.isna() | source.astype('string').str.strip().eq('')
        flag((~missing) & (converted.isna() | ~np.isfinite(converted)), f'invalid_{column}')
        frame[column] = converted
    flag(~frame.latitude.between(-90, 90), 'latitude_out_of_range')
    flag(~frame.longitude.between(-180, 180), 'longitude_out_of_range')
    if not allow_negative_depth:
        flag(frame.depth.notna() & frame.depth.lt(0), 'negative_depth')
    rejected = frame.loc[reasons.ne('')].copy()
    rejected['rejection_reason'] = reasons[reasons.ne('')]
    valid = frame.loc[reasons.eq('')].copy()
    # Keep half-open query scope even when the provider includes both endpoints.
    outside = pd.Series(False, index=valid.index)
    if start is not None:
        outside |= valid.event_time_utc.lt(parse_time(start, source_timezone))
    if end is not None:
        outside |= valid.event_time_utc.ge(parse_time(end, source_timezone))
    out_of_scope = int(outside.sum())
    valid = valid.loc[~outside]
    valid = valid.sort_values('updated_at', na_position='first', kind='stable')
    duplicate_rows = valid.loc[valid.event_id.duplicated(keep='last')].copy()
    valid = valid.drop_duplicates('event_id', keep='last')
    valid['province_key'] = valid.province.map(name_key)
    valid = valid.merge(provinces[['province_key', 'province_id']], on='province_key', how='left', validate='many_to_one')
    valid['province_id'] = valid.province_id.fillna(0).astype('int64')
    valid['date_key'] = valid.event_time_utc.dt.strftime('%Y%m%d').astype('int64')
    valid = valid.rename(columns={'depth': 'depth_km', 'type': 'magnitude_type', 'province': 'source_province'})
    fields = ['event_id', 'event_time_utc', 'date_key', 'province_id', 'latitude', 'longitude',
              'magnitude', 'magnitude_type', 'depth_km', 'country', 'source_province', 'location',
              'event_source', 'source_event_url']
    valid = valid[fields].sort_values(['event_time_utc', 'event_id']).reset_index(drop=True)
    quality = {'raw_rows': len(frame), 'rejected_rows': len(rejected), 'out_of_scope_rows': out_of_scope,
               'duplicate_rows_removed': len(duplicate_rows), 'accepted_events': len(valid),
               'unmatched_province_events': int(valid.province_id.eq(0).sum()),
               'missing_magnitude_events': int(valid.magnitude.isna().sum()),
               'missing_depth_events': int(valid.depth_km.isna().sum())}
    assert sum(quality[k] for k in ['rejected_rows', 'out_of_scope_rows', 'duplicate_rows_removed', 'accepted_events']) == len(frame)
    return valid, rejected, duplicate_rows, quality


def date_dimension(start, end):
    dates = pd.date_range(pd.Timestamp(start).date(), pd.Timestamp(end).date(), freq='D')
    return pd.DataFrame({'date_key': dates.strftime('%Y%m%d').astype(int), 'date': dates.strftime('%Y-%m-%d'),
                         'year': dates.year, 'month': dates.month, 'month_name': dates.month_name(),
                         'quarter': dates.quarter})


def validate_facilities(facilities, provinces, allow_synthetic=False):
    fields = ['facility_id', 'facility_name', 'facility_type', 'province_id', 'latitude', 'longitude',
              'source_url', 'verified_on', 'coordinate_quality', 'is_synthetic']
    if not set(fields).issubset(facilities.columns):
        raise ValueError('Facility input must match data/facilities_template.csv.')
    f = facilities[fields].copy()
    if f.empty:
        return f
    for c in ['facility_id', 'facility_name', 'facility_type', 'source_url', 'coordinate_quality']:
        if f[c].isna().any() or f[c].astype(str).str.strip().eq('').any():
            raise ValueError(f'Missing facility field: {c}')
    if f.facility_id.duplicated().any():
        raise ValueError('Duplicate facility IDs.')
    flags = f.is_synthetic.astype(str).str.lower().map({'true': True, 'false': False})
    if flags.isna().any() or (flags.any() and not allow_synthetic):
        raise ValueError('Synthetic facilities require explicit demo mode.')
    f['is_synthetic'] = flags
    for c, lower, upper in [('latitude', -90, 90), ('longitude', -180, 180)]:
        f[c] = pd.to_numeric(f[c], errors='coerce')
        if not f[c].between(lower, upper).all():
            raise ValueError(f'Invalid facility {c}')
    f['province_id'] = pd.to_numeric(f.province_id, errors='raise')
    if not f.province_id.isin(provinces.province_id).all():
        raise ValueError('Facility province foreign key missing.')
    if pd.to_datetime(f.verified_on, errors='coerce').isna().any():
        raise ValueError('Invalid verification date.')
    if not f.coordinate_quality.isin(['verified_site', 'approximate', 'synthetic']).all():
        raise ValueError('Unknown coordinate quality.')
    return f


def facility_distances(events, facilities, event_id):
    """All facilities for ONE selected event, not an unbounded cross join."""
    event = events.loc[events.event_id.eq(str(event_id))]
    if len(event) != 1:
        raise ValueError('Select one existing, unique event ID.')
    e = event.iloc[0]
    f = facilities.copy()
    lat1, lon1 = np.radians([float(e.latitude), float(e.longitude)])
    lat2, lon2 = np.radians(f.latitude.to_numpy(dtype=float)), np.radians(f.longitude.to_numpy(dtype=float))
    a = np.sin((lat2-lat1)/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin((lon2-lon1)/2)**2
    f['distance_km'] = 6371.0088 * 2 * np.arcsin(np.sqrt(np.clip(a, 0, 1)))
    f['event_id'] = str(event_id)
    return f[['event_id', 'facility_id', 'distance_km']].sort_values('distance_km').reset_index(drop=True)


def export_model(tables, quality, output_dir, metadata):
    """New snapshot per run; CSV for Power BI and SQLite for reproducible SQL."""
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=False)
    for name, df in tables.items():
        df.to_csv(output_dir / f'{name}.csv', index=False, encoding='utf-8-sig')
    with sqlite3.connect(output_dir / 'analytics.sqlite') as conn:
        for name, df in tables.items():
            df.to_sql(name, conn, if_exists='fail', index=False)
        for table, key in [('dim_province','province_id'), ('dim_date','date_key'),
                           ('dim_facility','facility_id'), ('fact_earthquake','event_id')]:
            conn.execute(f'CREATE UNIQUE INDEX uq_{table} ON {table}({key})')
        conn.execute('CREATE UNIQUE INDEX uq_distance ON fact_event_facility(event_id, facility_id)')
        for fk, dim, key in [('province_id','dim_province','province_id'), ('date_key','dim_date','date_key')]:
            missing = conn.execute(f'SELECT COUNT(*) FROM fact_earthquake f LEFT JOIN {dim} d ON f.{fk}=d.{key} WHERE d.{key} IS NULL').fetchone()[0]
            if missing:
                raise ValueError(f'Missing foreign keys: {fk}')
    manifest = {**metadata, 'created_at_utc': datetime.now(timezone.utc).isoformat(), 'quality': quality,
                'tables': {name: {'rows': len(df), 'sha256': hashlib.sha256((output_dir/f'{name}.csv').read_bytes()).hexdigest()} for name,df in tables.items()}}
    (output_dir/'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    return output_dir


## 2. Run configuration

The default live mode queries USGS for 6 February 2023, magnitude 4+, within 35–43 N / 25–45 E. This rectangle includes neighboring countries and sea areas; it is not a Turkey boundary. USGS timestamps and date keys are UTC. End is exclusive after normalization.

To run the offline quality-check example instead, change `MODE` to `"demo"`. AFAD returned empty results in the user's Colab checks, so the working live path uses USGS. Catalogue coverage differs by provider, especially for smaller events.


In [ ]:
MODE = "live"  # "live" (USGS) or "demo" (offline synthetic fixture)
START = datetime(2024, 9, 15) if MODE == "demo" else datetime(2023, 2, 6)
END = datetime(2024, 9, 17) if MODE == "demo" else datetime(2023, 2, 7)
LIVE_SOURCE_TIMEZONE = "UTC"  # USGS timestamps are UTC
MIN_MAGNITUDE = 4.0  # query scope, not a completeness threshold
SCREENING_RADIUS_KM = 100.0  # optional illustrative screening choice
SELECTED_EVENT_ID = "demo-001"  # used only by the optional synthetic facility example
ROOT = Path.cwd()
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
OUTPUT_DIR = ROOT / "outputs" / f"{MODE}_{RUN_ID}"
RAW_DIR = ROOT / "data" / "raw" / RUN_ID
if MODE not in {"demo", "live"}:
    raise ValueError("MODE must be demo or live")
if not math.isfinite(SCREENING_RADIUS_KM) or SCREENING_RADIUS_KM < 0:
    raise ValueError("Choose a finite, nonnegative screening radius")
print(f"Mode: {MODE.upper()} | Output: {OUTPUT_DIR}")


## 3. Ingest reference data and earthquake records

Demo records are fabricated and deliberately contain quality issues. Live mode uses USGS:
each daily window is checked against its count endpoint; large windows are subdivided.
Raw responses are archived and count mismatches fail visibly. Source IDs, URLs, magnitude
types, and UTC timestamps are retained. Provider updates between requests can require a rerun.


In [ ]:
if MODE == "demo":
    source_timezone = "UTC"  # fixture timestamps are defined as UTC
    province_payload = {"data": [
        {"id": 34, "name": "İstanbul", "region": {"en": "Marmara"},
         "districts": [{"id": 1, "name": "Demo district A"}]},
        {"id": 35, "name": "İzmir", "region": {"en": "Aegean"},
         "districts": [{"id": 2, "name": "Demo district B"}]}]}
    base = {"eventID": "demo-001", "date": "2024-09-15T08:00:00",
            "latitude": "40.8", "longitude": "28.8", "depth": "10",
            "magnitude": "4.1", "type": "ML", "province": "İstanbul",
            "country": "Türkiye", "location": "SYNTHETIC EVENT"}
    records = [base,
        {**base, "magnitude": "4.2", "lastUpdateDate": "2024-09-15T09:00:00"},
        {**base, "eventID": "demo-002", "date": "2024-09-16T09:00:00",
         "latitude": "38.4", "longitude": "27.1", "province": "İZMİR", "magnitude": "-0.2"},
        {**base, "eventID": "demo-bad", "latitude": "999"},
        {**base, "eventID": "demo-003", "province": None, "depth": None, "magnitude": None}]
else:
    source_timezone = "UTC"
    # No province lookup is guessed from USGS place descriptions.
    province_payload = None
    records = fetch_usgs_events(START, END, RAW_DIR, min_magnitude=MIN_MAGNITUDE)
    if not records:
        raise RuntimeError("No USGS records for this scope; inspect the query before proceeding.")
    SELECTED_EVENT_ID = max(records, key=lambda r: float(r["magnitude"]) if r["magnitude"] is not None else float("-inf"))["eventID"]
print(f"Raw event rows: {len(records)} | Source: {'synthetic' if MODE == 'demo' else 'USGS'}")


## 4. Validate and normalize

Missing event IDs, invalid timestamps, bad coordinates, malformed numeric measurements,
are quarantined. Negative depths are permitted in the USGS path because its documented range includes them. Missing depth/magnitude remains missing. Negative
magnitudes are allowed. Duplicate event IDs resolve to the latest source update, with
last ingestion order breaking ties. Conflicting records remain inspectable in the raw
snapshots and duplicate audit. Unmatched source province names get key 0.

Province labels are the provider's labels, not a point-in-polygon boundary calculation.
Events outside Turkey are retained and can be filtered using the source country.
Population is deliberately not copied without a reference year or repeated on event rows.

USGS does not provide Turkish province IDs here. Live events keep province key 0 until a verified spatial lookup is added. Source place text is not used to guess provinces.


In [ ]:
if MODE == "demo":
    dim_province, dim_district = province_tables(province_payload)
else:
    dim_province = pd.DataFrame([{"province_id": 0, "province_name": "Unmatched / outside reference", "province_key": "", "region": None}])
    dim_district = pd.DataFrame(columns=["district_id", "district_name", "province_id"])
fact_earthquake, rejected_events, duplicate_events, quality = normalize_events(
    records, dim_province, source_timezone, START, END, allow_negative_depth=MODE == "live")
utc_start = parse_time(START, source_timezone)
utc_end = parse_time(END, source_timezone)
dim_date = date_dimension(utc_start, utc_end)
display(pd.DataFrame(quality.items(), columns=["Metric", "Value"]))
display(fact_earthquake)
display(rejected_events[["eventID", "rejection_reason"]])


## 5. Facility reference and proximity screening

For live mode, populate `data/facilities_template.csv` with verified public information.
Never replace missing site coordinates with a province center and call it a verified site.
Record provenance, verification date, coordinate quality, and synthetic status.

The calculation uses great-circle epicentral distance, not road distance or shaking
intensity. The bridge contains all mapped facilities for **one selected event**, allowing
the radius to change in Power BI without recalculating distances. Larger event selections
should be processed in controlled batches rather than an unbounded Cartesian join.


In [ ]:
facility_columns = ["facility_id", "facility_name", "facility_type", "province_id",
                    "latitude", "longitude", "source_url", "verified_on", "coordinate_quality", "is_synthetic"]
if MODE == "demo":
    facility_input = pd.DataFrame([
        ["demo-site-A", "Synthetic warehouse A", "Warehouse", 34, 40.9, 28.9,
         "synthetic://fixture", "2024-09-15", "synthetic", True],
        ["demo-site-B", "Synthetic store B", "Store", 35, 38.4, 27.1,
         "synthetic://fixture", "2024-09-15", "synthetic", True]], columns=facility_columns)
else:
    facility_path = ROOT / "data" / "facilities_template.csv"
    facility_input = pd.read_csv(facility_path, dtype={"facility_id": "string"}) if facility_path.exists() else pd.DataFrame(columns=facility_columns)
    if not facility_input.empty:
        # Verified reference data is needed to validate actual facility province IDs.
        province_payload = fetch_json(PROVINCE_URL, {"limit": 100}, RAW_DIR)
        dim_province, dim_district = province_tables(province_payload)
        if len(dim_province) != 82:
            raise ValueError("Expected unknown row + 81 verified province reference entries")
dim_facility = validate_facilities(facility_input, dim_province, allow_synthetic=MODE == "demo")
fact_event_facility = pd.DataFrame(columns=["event_id", "facility_id", "distance_km"])
if not dim_facility.empty:
    fact_event_facility = facility_distances(fact_earthquake, dim_facility, SELECTED_EVENT_ID)
    screening = fact_event_facility.merge(dim_facility, on="facility_id", validate="many_to_one")
    screening["in_screening_radius"] = screening.distance_km.le(SCREENING_RADIUS_KM)
    display(screening[["facility_name", "distance_km", "in_screening_radius", "coordinate_quality", "is_synthetic"]])
    print(f"{int(screening.in_screening_radius.sum())} of {len(screening)} mapped facilities within {SCREENING_RADIUS_KM:g} km")
else:
    print("Facility assessment unavailable: add verified locations. No zero-exposure claim is made.")


## 6. Export an auditable analytical model

Each run creates a new directory with typed-source CSVs, a SQLite database for local SQL,
a quality summary, and SHA-256 hashes. Dates use UTC. CSV consumers must apply types;
the Power Query example in `POWER_BI.md` shows how. Azure SQL deployment is a future step,
not an implemented feature of this notebook. No credentials are stored here.


In [ ]:
tables = {"dim_province": dim_province, "dim_district": dim_district, "dim_date": dim_date,
          "dim_facility": dim_facility, "fact_earthquake": fact_earthquake,
          "fact_event_facility": fact_event_facility}
export_model(tables, quality, OUTPUT_DIR, {
    "mode": MODE, "event_source": "synthetic" if MODE == "demo" else "USGS",
    "geographic_scope": "bbox 35..43 N, 25..45 E; includes neighboring areas",
    "minimum_magnitude": MIN_MAGNITUDE, "start_inclusive": START.isoformat(), "end_exclusive": END.isoformat(),
    "source_timezone": source_timezone, "date_key_timezone": "UTC",
    "selected_event_id": SELECTED_EVENT_ID, "screening_radius_km": SCREENING_RADIUS_KM,
    "facility_count": len(dim_facility), "live_api_verified_in_this_environment": False})
rejected_events.to_csv(OUTPUT_DIR / "rejected_events.csv", index=False)
duplicate_events.to_csv(OUTPUT_DIR / "duplicate_events.csv", index=False)
print(f"Saved {len(tables)} analytical tables, audit records, and manifest to {OUTPUT_DIR}")


## 7. SQL analysis: event activity by source location

USGS place descriptions are source labels rather than verified Turkish province assignments. This query summarizes events by those labels and magnitude type, and avoids presenting unassigned events as province-level findings.


In [ ]:
query = """
WITH activity AS (
 SELECT COALESCE(location, 'Location not supplied') AS source_location,
        COALESCE(magnitude_type, 'Unknown') AS magnitude_type,
        COUNT(*) AS recorded_events,
        MAX(magnitude) AS max_magnitude,
        AVG(depth_km) AS average_depth_km
 FROM fact_earthquake
 GROUP BY COALESCE(location, 'Location not supplied'),
          COALESCE(magnitude_type, 'Unknown')
)
SELECT source_location, magnitude_type, recorded_events, max_magnitude,
       ROUND(average_depth_km, 1) AS average_depth_km,
       DENSE_RANK() OVER (PARTITION BY magnitude_type ORDER BY recorded_events DESC) AS activity_rank
FROM activity
ORDER BY activity_rank, max_magnitude DESC
"""
with sqlite3.connect(OUTPUT_DIR / "analytics.sqlite") as conn:
    display(pd.read_sql_query(query, conn))


## 8. Completion checks

The live run exports real USGS catalogue events for the documented query window. The offline demo checks the pipeline with fabricated records. In either mode, inspect the map coordinates, event timeline, magnitude and depth fields, and quality summary before importing the CSVs into Power BI.

Sources:
- USGS Earthquake Catalog API: https://earthquake.usgs.gov/fdsnws/event/1/
- AFAD API documentation (explored; returned empty records in initial checks): https://deprem.afad.gov.tr/event-service
- Türkiye province reference used only in the optional demo: https://ubeydeozdmr.github.io/turkiye-api-templates/docs/


In [ ]:
if MODE == "demo":
    assert quality["raw_rows"] == 5
    assert quality["rejected_rows"] == 1
    assert quality["duplicate_rows_removed"] == 1
    assert quality["accepted_events"] == 3
    assert int(screening.in_screening_radius.sum()) == 1
assert fact_earthquake.event_id.is_unique
assert fact_earthquake.province_id.isin(dim_province.province_id).all()
assert fact_earthquake.date_key.isin(dim_date.date_key).all()
print("Reconciliation and model-key checks passed.")
